In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import random
import scipy
import plotly
import openpyxl
import psycopg2

print("All libraries imported successfully ✅")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

All libraries imported successfully ✅
Pandas version: 2.2.3
Numpy version: 2.1.3


In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

fake = Faker('en_IN')
np.random.seed(42)
random.seed(42)

print("Imports done ✅")

Imports done ✅


In [3]:
# Indian Cities by Tier
tier1_cities = [
    'Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai',
    'Kolkata', 'Pune', 'Ahmedabad'
]

tier2_cities = [
    'Jaipur', 'Lucknow', 'Kanpur', 'Nagpur', 'Indore',
    'Bhopal', 'Visakhapatnam', 'Patna', 'Vadodara', 'Surat',
    'Coimbatore', 'Agra', 'Madurai', 'Nashik', 'Ranchi'
]

tier3_cities = [
    'Meerut', 'Faridabad', 'Rajkot', 'Varanasi', 'Aurangabad',
    'Amritsar', 'Allahabad', 'Jodhpur', 'Guwahati', 'Chandigarh',
    'Mysore', 'Jabalpur', 'Tiruchirappalli', 'Bareilly', 'Aligarh',
    'Moradabad', 'Gorakhpur', 'Bikaner', 'Noida', 'Ghaziabad'
]

# UPI Platforms
platforms = ['PhonePe', 'Google Pay', 'Paytm', 'CRED', 'Amazon Pay', 'WhatsApp Pay']
platform_weights = [0.35, 0.30, 0.18, 0.08, 0.05, 0.04]

# Merchant Categories with realistic transaction amount ranges (min, max, avg)
merchant_categories = {
    'Food & Beverage':      (20, 2500, 250),
    'Grocery':              (50, 5000, 800),
    'Fuel':                 (100, 5000, 1500),
    'Rent':                 (2000, 50000, 12000),
    'Utilities':            (100, 10000, 1200),
    'E-commerce':           (99, 25000, 1800),
    'Healthcare':           (50, 15000, 900),
    'Education':            (500, 50000, 3500),
    'Entertainment':        (50, 2000, 400),
    'Travel':               (200, 30000, 3000),
    'Clothing & Fashion':   (199, 10000, 1500),
    'Electronics':          (500, 80000, 8000),
    'P2P Transfer':         (50, 100000, 2500),
    'Insurance':            (500, 25000, 4000),
    'Investment':           (100, 500000, 15000),
    'Salon & Wellness':     (100, 3000, 600),
    'Auto & Transport':     (20, 2000, 180),
    'Hotel & Lodging':      (500, 20000, 3500),
}

# Payment modes
payment_modes = ['UPI ID', 'QR Scan', 'Payment Link', 'Intent']
payment_mode_weights = [0.40, 0.35, 0.15, 0.10]

# Device types
device_types = ['Android', 'iOS', 'Web']
device_weights = [0.72, 0.20, 0.08]

# Transaction status
status_options = ['Success', 'Failed', 'Pending']
status_weights = [0.91, 0.06, 0.03]

# Gender
genders = ['Male', 'Female', 'Other']
gender_weights = [0.54, 0.44, 0.02]

print("All reference data defined ✅")
print(f"Total merchant categories: {len(merchant_categories)}")
print(f"Total platforms: {len(platforms)}")
print(f"Tier 1 cities: {len(tier1_cities)}")
print(f"Tier 2 cities: {len(tier2_cities)}")
print(f"Tier 3 cities: {len(tier3_cities)}")

All reference data defined ✅
Total merchant categories: 18
Total platforms: 6
Tier 1 cities: 8
Tier 2 cities: 15
Tier 3 cities: 20


In [4]:
# Generate 10,000 unique users
NUM_USERS = 10000

def generate_users(n):
    users = []
    for i in range(n):
        # Assign city and tier
        city_tier = random.choices(
            ['Tier1', 'Tier2', 'Tier3'],
            weights=[0.35, 0.40, 0.25]
        )[0]
        
        if city_tier == 'Tier1':
            city = random.choice(tier1_cities)
        elif city_tier == 'Tier2':
            city = random.choice(tier2_cities)
        else:
            city = random.choice(tier3_cities)
        
        # Age distribution — realistic for UPI users in India
        age = int(np.random.choice(
            range(18, 65),
            p=np.array(
                [3]*5 +    # 18-22: students
                [5]*8 +    # 23-30: young professionals
                [4]*10 +   # 31-40: mid career
                [3]*10 +   # 41-50: senior
                [1]*14     # 51-64: older users
            ) / np.array(
                [3]*5 +
                [5]*8 +
                [4]*10 +
                [3]*10 +
                [1]*14
            ).sum()
        ))
        
        # Platform preference
        platform = random.choices(platforms, weights=platform_weights)[0]
        
        # Monthly transaction frequency (how active is this user)
        activity_level = random.choices(
            ['Low', 'Medium', 'High', 'Power'],
            weights=[0.20, 0.40, 0.30, 0.10]
        )[0]
        
        freq_map = {
            'Low': random.randint(1, 5),
            'Medium': random.randint(6, 20),
            'High': random.randint(21, 50),
            'Power': random.randint(51, 150)
        }
        monthly_txn_freq = freq_map[activity_level]
        
        users.append({
            'user_id': f'USR{str(i+1).zfill(6)}',
            'age': age,
            'gender': random.choices(genders, weights=gender_weights)[0],
            'city': city,
            'city_tier': city_tier,
            'preferred_platform': platform,
            'activity_level': activity_level,
            'monthly_txn_frequency': monthly_txn_freq,
            'device_type': random.choices(device_types, weights=device_weights)[0],
            'registration_date': fake.date_between(
                start_date=datetime(2019, 1, 1),
                end_date=datetime(2022, 12, 31)
            )
        })
    
    return pd.DataFrame(users)

df_users = generate_users(NUM_USERS)

print(f"Users generated: {len(df_users)} ✅")
print(f"\nActivity Level Distribution:")
print(df_users['activity_level'].value_counts())
print(f"\nCity Tier Distribution:")
print(df_users['city_tier'].value_counts())
print(f"\nTop Platforms:")
print(df_users['preferred_platform'].value_counts())

Users generated: 10000 ✅

Activity Level Distribution:
activity_level
Medium    4007
High      2957
Low       2044
Power      992
Name: count, dtype: int64

City Tier Distribution:
city_tier
Tier2    4095
Tier1    3429
Tier3    2476
Name: count, dtype: int64

Top Platforms:
preferred_platform
PhonePe         3500
Google Pay      3015
Paytm           1822
CRED             771
Amazon Pay       478
WhatsApp Pay     414
Name: count, dtype: int64


In [5]:
# Generate 2,000 unique merchants
NUM_MERCHANTS = 2000

def generate_merchants(n):
    merchants = []
    
    # Merchant name templates by category
    name_templates = {
        'Food & Beverage':      ['Cafe', 'Dhaba', 'Restaurant', 'Biryani House', 'Sweet Shop', 'Juice Corner', 'Tiffin Center'],
        'Grocery':              ['Kirana Store', 'Supermart', 'Fresh Mart', 'Daily Needs', 'General Store'],
        'Fuel':                 ['Petrol Pump', 'Fuel Station', 'HP Petrol', 'Indian Oil', 'BPCL Outlet'],
        'Rent':                 ['Properties', 'Realty', 'Housing', 'Residency', 'Apartments'],
        'Utilities':            ['Electric Board', 'Water Supply', 'Gas Agency', 'Broadband', 'DTH Services'],
        'E-commerce':           ['Amazon', 'Flipkart', 'Meesho', 'Myntra', 'Nykaa', 'Snapdeal'],
        'Healthcare':           ['Pharmacy', 'Clinic', 'Hospital', 'Diagnostic Lab', 'Medical Store'],
        'Education':            ['Coaching Center', 'School Fees', 'College', 'Tuition', 'Academy'],
        'Entertainment':        ['Cinema', 'OTT Platform', 'Gaming Zone', 'Multiplex', 'Event'],
        'Travel':               ['Airlines', 'Railway Booking', 'Bus Service', 'Travel Agency', 'Cab Service'],
        'Clothing & Fashion':   ['Boutique', 'Garments', 'Fashion Store', 'Textile', 'Readymade'],
        'Electronics':          ['Electronics', 'Mobile Store', 'Gadget Shop', 'Tech Store', 'Appliances'],
        'P2P Transfer':         ['Personal Transfer', 'Family Transfer', 'Friend Transfer'],
        'Insurance':            ['LIC', 'Insurance Co', 'Policy Bazaar', 'Health Insurance', 'Motor Insurance'],
        'Investment':           ['Mutual Fund', 'Stock Broker', 'Zerodha', 'Groww', 'SIP Payment'],
        'Salon & Wellness':     ['Salon', 'Spa', 'Beauty Parlour', 'Gym', 'Wellness Center'],
        'Auto & Transport':     ['Auto Rickshaw', 'Ola', 'Uber', 'Rapido', 'Metro Card'],
        'Hotel & Lodging':      ['Hotel', 'Lodge', 'Guest House', 'Inn', 'Service Apartment'],
    }
    
    # Category distribution — food and grocery most common merchants
    category_weights = [
        0.15, 0.12, 0.06, 0.04, 0.05,
        0.08, 0.06, 0.04, 0.05, 0.05,
        0.05, 0.04, 0.06, 0.03, 0.03,
        0.04, 0.08, 0.07
    ]
    
    categories = list(name_templates.keys())
    
    for i in range(n):
        category = random.choices(categories, weights=category_weights)[0]
        
        # Assign merchant city
        city_tier = random.choices(
            ['Tier1', 'Tier2', 'Tier3'],
            weights=[0.40, 0.40, 0.20]
        )[0]
        
        if city_tier == 'Tier1':
            city = random.choice(tier1_cities)
        elif city_tier == 'Tier2':
            city = random.choice(tier2_cities)
        else:
            city = random.choice(tier3_cities)
        
        template = random.choice(name_templates[category])
        merchant_name = f"{fake.last_name()} {template}"
        
        merchants.append({
            'merchant_id': f'MER{str(i+1).zfill(5)}',
            'merchant_name': merchant_name,
            'merchant_category': category,
            'merchant_city': city,
            'merchant_city_tier': city_tier,
            'is_active': random.choices([1, 0], weights=[0.92, 0.08])[0]
        })
    
    return pd.DataFrame(merchants)

df_merchants = generate_merchants(NUM_MERCHANTS)

print(f"Merchants generated: {len(df_merchants)} ✅")
print(f"\nTop Merchant Categories:")
print(df_merchants['merchant_category'].value_counts().head(10))
print(f"\nMerchant City Tier Distribution:")
print(df_merchants['merchant_city_tier'].value_counts())

Merchants generated: 2000 ✅

Top Merchant Categories:
merchant_category
Food & Beverage       276
Grocery               218
E-commerce            178
Auto & Transport      135
Hotel & Lodging       125
Healthcare            118
Fuel                  105
Utilities             100
Clothing & Fashion     95
Travel                 93
Name: count, dtype: int64

Merchant City Tier Distribution:
merchant_city_tier
Tier1    821
Tier2    817
Tier3    362
Name: count, dtype: int64


In [6]:
# Generate 500,000 transactions
NUM_TRANSACTIONS = 500000

def generate_transactions(n, df_users, df_merchants):
    
    transactions = []
    
    # Date range: Jan 2022 to Dec 2023 (2 full years)
    start_date = datetime(2022, 1, 1)
    end_date = datetime(2023, 12, 31)
    date_range = (end_date - start_date).days
    
    # Festival months — higher transaction volumes
    festival_months = {
        1: 1.1,   # New Year
        3: 1.1,   # Holi
        8: 1.2,   # Independence Day / Raksha Bandhan
        9: 1.3,   # Navratri / Onam
        10: 1.5,  # Dussehra / Durga Puja
        11: 1.8,  # Diwali — highest spike
        12: 1.3,  # Christmas / Year End
    }
    
    user_ids = df_users['user_id'].tolist()
    merchant_ids = df_merchants['merchant_id'].tolist()
    merchant_lookup = df_merchants.set_index('merchant_id').to_dict('index')
    user_lookup = df_users.set_index('user_id').to_dict('index')
    
    for i in range(n):
        # Pick a random user
        user_id = random.choice(user_ids)
        user = user_lookup[user_id]
        
        # Pick a random merchant
        merchant_id = random.choice(merchant_ids)
        merchant = merchant_lookup[merchant_id]
        
        # Generate transaction date
        txn_date = start_date + timedelta(days=random.randint(0, date_range))
        month = txn_date.month
        
        # Generate transaction hour — realistic patterns
        # Peak hours: 9-11am, 1-3pm, 7-10pm
        hour_weights = [
            0.5, 0.3, 0.2, 0.2, 0.3, 0.5,   # 0-5 (late night — low)
            1.0, 2.0, 3.5, 4.0, 3.5, 3.0,   # 6-11 (morning peak)
            3.5, 4.0, 3.5, 3.0, 2.5, 3.0,   # 12-17 (afternoon)
            3.5, 4.5, 4.0, 3.0, 2.0, 1.0    # 18-23 (evening peak)
        ]
        hour = random.choices(range(24), weights=hour_weights)[0]
        minute = random.randint(0, 59)
        second = random.randint(0, 59)
        
        txn_datetime = txn_date.replace(hour=hour, minute=minute, second=second)
        
        # Generate amount based on merchant category
        cat = merchant['merchant_category']
        min_amt, max_amt, avg_amt = merchant_categories[cat]
        
        # Use log-normal distribution for realistic amount skew
        amount = round(np.random.lognormal(
            mean=np.log(avg_amt),
            sigma=0.6
        ), 2)
        amount = max(min_amt, min(amount, max_amt))
        
        # Festival multiplier on amount
        festival_multiplier = festival_months.get(month, 1.0)
        if random.random() < 0.3:
            amount = round(amount * festival_multiplier, 2)
        
        # Payment mode
        payment_mode = random.choices(payment_modes, weights=payment_mode_weights)[0]
        
        # Transaction status
        status = random.choices(status_options, weights=status_weights)[0]
        
        # --- FRAUD SIGNAL GENERATION ---
        
        # Flag 1: Unusual hour (transactions between 1am - 5am)
        unusual_hour_flag = 1 if hour in range(1, 5) else 0
        
        # Flag 2: High amount anomaly (top 2% amounts in that category)
        high_amount_flag = 1 if amount > (avg_amt * 4) else 0
        
        # Flag 3: Round amount flag (fraud often uses round numbers)
        round_amount_flag = 1 if amount % 1000 == 0 and amount >= 5000 else 0
        
        # Flag 4: Velocity flag (simulated — user making txn within same minute)
        velocity_flag = 1 if random.random() < 0.02 else 0
        
        # Combined fraud score (0-4)
        fraud_score = unusual_hour_flag + high_amount_flag + round_amount_flag + velocity_flag
        
        # Is fraud — if fraud score >= 2 AND random trigger
        is_fraud = 1 if (fraud_score >= 2 and random.random() < 0.35) else 0
        
        # Platform used for this transaction
        # 80% chance user uses their preferred platform
        if random.random() < 0.80:
            platform_used = user['preferred_platform']
        else:
            platform_used = random.choices(platforms, weights=platform_weights)[0]
        
        transactions.append({
            'txn_id':               f'TXN{str(i+1).zfill(8)}',
            'txn_datetime':         txn_datetime,
            'txn_date':             txn_date.date(),
            'txn_month':            month,
            'txn_year':             txn_date.year,
            'txn_hour':             hour,
            'day_of_week':          txn_date.strftime('%A'),
            'user_id':              user_id,
            'user_city':            user['city'],
            'user_city_tier':       user['city_tier'],
            'user_age':             user['age'],
            'user_gender':          user['gender'],
            'user_activity_level':  user['activity_level'],
            'merchant_id':          merchant_id,
            'merchant_name':        merchant['merchant_name'],
            'merchant_category':    cat,
            'merchant_city':        merchant['merchant_city'],
            'platform_used':        platform_used,
            'preferred_platform':   user['preferred_platform'],
            'is_preferred_platform':1 if platform_used == user['preferred_platform'] else 0,
            'payment_mode':         payment_mode,
            'device_type':          user['device_type'],
            'amount':               amount,
            'status':               status,
            'unusual_hour_flag':    unusual_hour_flag,
            'high_amount_flag':     high_amount_flag,
            'round_amount_flag':    round_amount_flag,
            'velocity_flag':        velocity_flag,
            'fraud_score':          fraud_score,
            'is_fraud':             is_fraud,
        })
        
        if (i+1) % 100000 == 0:
            print(f"  Generated {i+1:,} transactions...")
    
    return pd.DataFrame(transactions)

print("Generating 500,000 transactions... this will take 2-3 minutes ⏳")
df_transactions = generate_transactions(NUM_TRANSACTIONS, df_users, df_merchants)

print(f"\n✅ Transactions generated: {len(df_transactions):,}")
print(f"\nStatus Distribution:")
print(df_transactions['status'].value_counts())
print(f"\nFraud Cases: {df_transactions['is_fraud'].sum():,} ({df_transactions['is_fraud'].mean()*100:.2f}%)")
print(f"\nTop Merchant Categories by Volume:")
print(df_transactions['merchant_category'].value_counts().head(5))
print(f"\nDate Range: {df_transactions['txn_date'].min()} to {df_transactions['txn_date'].max()}")

Generating 500,000 transactions... this will take 2-3 minutes ⏳
  Generated 100,000 transactions...
  Generated 200,000 transactions...
  Generated 300,000 transactions...
  Generated 400,000 transactions...
  Generated 500,000 transactions...

✅ Transactions generated: 500,000

Status Distribution:
status
Success    454849
Failed      30164
Pending     14987
Name: count, dtype: int64

Fraud Cases: 247 (0.05%)

Top Merchant Categories by Volume:
merchant_category
Food & Beverage     68520
Grocery             54568
E-commerce          44577
Auto & Transport    33841
Hotel & Lodging     31458
Name: count, dtype: int64

Date Range: 2022-01-01 to 2023-12-31


In [8]:
# Save all three datasets to the raw data folder
import os

# Set your path — update this to match your exact folder location
raw_path = r"D:\Projects\End-to-end projects\10. UPI Payments Intelligence\Data\Raw"

# Save transactions
df_transactions.to_csv(os.path.join(raw_path, 'upi_transactions.csv'), index=False)
print(f"✅ Transactions saved: {len(df_transactions):,} rows")

# Save users
df_users.to_csv(os.path.join(raw_path, 'upi_users.csv'), index=False)
print(f"✅ Users saved: {len(df_users):,} rows")

# Save merchants
df_merchants.to_csv(os.path.join(raw_path, 'upi_merchants.csv'), index=False)
print(f"✅ Merchants saved: {len(df_merchants):,} rows")

print(f"\nAll raw data saved successfully ✅")
print(f"\nFile sizes:")
for f in ['upi_transactions.csv', 'upi_users.csv', 'upi_merchants.csv']:
    size = os.path.getsize(os.path.join(raw_path, f)) / (1024*1024)
    print(f"  {f}: {size:.2f} MB")

✅ Transactions saved: 500,000 rows
✅ Users saved: 10,000 rows
✅ Merchants saved: 2,000 rows

All raw data saved successfully ✅

File sizes:
  upi_transactions.csv: 99.88 MB
  upi_users.csv: 0.66 MB
  upi_merchants.csv: 0.11 MB
